# Musanga V4 — NB07 v4_006 | Seam-suppressed RGB+MS inference and reference atlas

**Changes:** Apply overlap-context / halo inference (64 px on each side of every 256 px *output* tile). The U-Net predicts from a larger scene; only the central output is saved. At RGB mosaic edges, safely pad to a multiple of 32, preserving core geolocation. Prediction file names include `halo064` so older unpadded tiles are **never silently reused**. Build clean two-band GeoTIFF profiles without incompatible block-size parameters.

**Atlas:** six panels: RGB, MS false colour, RGB + annotations, probability + annotations, binary + annotations, strong red-on-RGB prediction overlay + annotations. Draw **actual individual reference crown polygons** as thick white lines with a thin black casing. Yellow denotes the reference footprint boundary only. Panel colour styling does not alter pixel probabilities or annotations.

**Objective spatial/sensor policy unchanged:** valid RGB, valid DSM and **all four MS bands from the same flight**, and the centre pixel must lie at least **100 m inside ROI**. DSM is a coverage gate (7-channel model is RGB+MS). No full mosaic scans; CPU blocks 00–03 read headers only, GPU block 04 streams native-resolution tiles.

**Pilot:** `SITE='Baego'`, `MAX_TILES=64` prioritizes circles/plots. Then use `SITE='ALL'`, `MAX_TILES=None` for production only after reviewing actual images. This is exploratory inference from the three existing NB05 folds; not additional held-out validation.

## 00 — Input registry and reproducible objective inference settings

In [ ]:
%pip -q install rasterio geopandas pyogrio shapely scipy pandas matplotlib segmentation-models-pytorch
from pathlib import Path
from contextlib import ExitStack
from collections import defaultdict
import os,json,hashlib,gc,warnings,math
import numpy as np,pandas as pd,geopandas as gpd,matplotlib.pyplot as plt
import rasterio
from rasterio.windows import Window,from_bounds,transform as window_transform
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling
from rasterio.transform import Affine
from rasterio.features import rasterize
from rasterio.warp import reproject
from scipy.ndimage import distance_transform_edt
from shapely.geometry import box
from shapely.ops import unary_union
from IPython.display import display
from google.colab import drive
if not Path('/content/drive/MyDrive').exists(): drive.mount('/content/drive')
ROOT=next((p for p in (Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final'),
                        Path('/content/drive/MyDrive/Musanga/Final')) if p.is_dir()),None)
assert ROOT is not None
DATA=ROOT/'Data';V4=ROOT/'Outputs/v4';AUDIT=V4/'00_input_audit'
OLD=ROOT/'Outputs/V3_1'; NB05=V4/'05_rgb_dsm_ms_matched_v1'
MODEL_DIR=NB05/'04_fusion_models'
OUT=V4/'07_RGB_MS_100m_ROI_complete_sensors_v4'
QA_DIR=OUT/'quality_maps'; TILE_DIR=OUT/'prediction_tiles';FIG=OUT/'figures'
for p in (QA_DIR,TILE_DIR,FIG):p.mkdir(parents=True,exist_ok=True)
V31=json.loads((OLD/'00_input_audit/00_input_manifest.json').read_text())
F78=json.loads((AUDIT/'00_v4_field78_contract.json').read_text())
TRAIN=json.loads((V4/'03_paired_rgb_unet_v1/03_training_contract.json').read_text())
REG=pd.read_csv(NB05/'05_harmonized_sensor_registry.csv')
assert {'group','band','path','data_band'}.issubset(REG.columns)
MOSAICS={}
for v in V31['mosaics']:
 if str(v['file']).endswith('.tif'):
  p=Path(v['path'])
  if not p.is_file():p=DATA/'Ortho_RGB'/v['file']
  assert p.is_file(),p
  MOSAICS[v['file']]=p
assert len(MOSAICS)==6
# The exact same-flight MS registry built by NB05 (including Yoko P1/P2).
GROUP_MOSAIC={
 'Baego_MS':'Baego.tif',
 'MOS_1_2_MS':'Mos_1_2_transparent_mosaic_group1_georef.tif',
 'MOS_3_4_MS':'Mos_3_4_transparent_mosaic_group1_georef.tif',
 'MOS_5_6_MS':'Mos_5_6_transparent_mosaic_group1_georef.tif',
 'Yangambi_YBJoseph_right_MS':'Yangambi_georef.tif',
 'Yangambi_YBForgeo_right_MS':'Yangambi_georef.tif'}
for group in REG.group.unique():
 if str(group).startswith('Yoko_Carlier2020_MicaSense_'):
  yoko=[m for m in MOSAICS if 'yoko' in m.lower()]
  assert len(yoko)==1,yoko
  GROUP_MOSAIC[group]=yoko[0]
assert set(REG.group.unique())==set(GROUP_MOSAIC),(
 'MS registry changed since NB05',set(REG.group.unique())^set(GROUP_MOSAIC))
ROI=gpd.read_file(Path(V31['roi'])).to_crs('EPSG:32635')
assert ROI.crs.is_projected
ROI_FULL=unary_union(ROI.geometry)
assert ROI_FULL.is_valid and not ROI_FULL.is_empty
SITE='Baego'   # 'Baego', 'MOS', 'Yangambi', 'Yoko', 'ALL'
ROI_SETBACK_M=100.0
ROI_INTERIOR=ROI_FULL.buffer(-ROI_SETBACK_M,join_style=1)
assert not ROI_INTERIOR.is_empty,'100 m inward buffer removed the complete ROI'
QA_RES_M=20.0   # coarse diagnostic only; inference uses native RGB pixels
MAX_TILES=64    # Reference-centred GPU pilot; None for all eligible tiles
EDGE_HALO_PX=64  # context beyond each 256px output tile; 64 makes 384px input
assert EDGE_HALO_PX>=0 and EDGE_HALO_PX%32==0
PRIORITIZE_REFERENCE_TILES=True
MAX_PRIORITY_FOOTPRINTS_PER_MOSAIC=12
QA_BLOCK=256    # maximum coarse QA-array dimensions per raster read
TILE=256
MIN_TILE_QA_FRACTION=0.01  # Efficiency only; not a quality threshold
PROB_THRESHOLD=.5
assert SITE in ('Baego','MOS','Yangambi','Yoko','ALL')
def site_of(m):
 return ('Baego' if m.startswith('Baego') else 'MOS' if m.startswith('Mos_')
         else 'Yangambi' if m.startswith('Yangambi') else 'Yoko')
SELECTED=[m for m in MOSAICS if SITE=='ALL' or site_of(m)==SITE]
def bands_for(group):
 q=REG[REG.group.eq(group)]
 assert len(q)==4 and set(q.band)=={'red','green','red_edge','nir'},group
 return {b:(Path(q[q.band.eq(b)].iloc[0].path),
            int(q[q.band.eq(b)].iloc[0].data_band))
         for b in ('red','green','red_edge','nir')}
for group in GROUP_MOSAIC:
 for p,_ in bands_for(group).values():assert p.is_file(),p
PAIRS=pd.read_csv(AUDIT/'00_rgb_dsm_pair_audit.csv')
DSM_BY_MOSAIC={Path(z.rgb_path).name:Path(z.dsm_path) for z in PAIRS.itertuples(index=False)}
assert set(MOSAICS)==set(DSM_BY_MOSAIC)
assert all(p.is_file() for p in DSM_BY_MOSAIC.values())
print('Mosaics:',SELECTED,'| ROI setback:',ROI_SETBACK_M,'m | outputs:',OUT)


## 01 — Fast source audit (headers ONLY; no resampling, no pixel reading)

The previous QA overview was unnecessary and could consume excessive RAM and minutes of I/O. This section now checks only raster metadata and records which RGB/DSM/MS flight sources exist. Sensor support is actually checked in Section 04 at the native RGB pixel scale.

In [ ]:
# No raster read() or WarpedVRT; only source headers and metadata.
# Keep a tiny registry, rather than constructing an enormous preview raster.
record=[]
for mosaic in SELECTED:
 with rasterio.open(MOSAICS[mosaic]) as rgb, rasterio.open(DSM_BY_MOSAIC[mosaic]) as dsm:
  assert rgb.crs is not None and dsm.crs is not None
  assert rgb.count>=3 and all(np.dtype(t)==np.dtype('uint8') for t in rgb.dtypes[:3])
  flights=[g for g,m in GROUP_MOSAIC.items() if m==mosaic]
  assert flights,('No complete MS source registered',mosaic)
  for group in flights:
   for band,(p,idx) in bands_for(group).items():
    with rasterio.open(p) as src:
     assert src.crs is not None and 1<=idx<=src.count,(group,band,p)
  record.append(dict(mosaic=mosaic,flight=';'.join(flights),
    qa_path='',roi_interior_pixels=np.nan,accepted_fraction=np.nan,
    rgb_resolution_m=abs(rgb.transform.a),
    rgb_width=rgb.width,rgb_height=rgb.height,
    dsm_resolution_m=abs(dsm.transform.a),n_four_band_MS_flights=len(flights)))
QA=pd.DataFrame(record)
QA.to_csv(OUT/'01_source_header_audit.csv',index=False)
display(QA[['mosaic','flight','rgb_resolution_m','n_four_band_MS_flights']])
print('Header checks done; no imagery loaded. Proceed to inference.')


## 02 — Optional: frozen reference inventory (no source-image reads)

We can revisit reference coverage once inference outputs exist. This inventory is not required to run the model.

In [ ]:
UNITS=gpd.read_file(V4/'01_spatial_cover_folds_v2/01_reference_groups_and_folds_v2.gpkg')
UNITS['unit_id']=UNITS.unit_id.astype(str)
OWNERS=pd.read_csv(V4/'02_rgb_circles_plus_plots_v3/02_reference_mosaic_owners_v2.csv')
OWNERS['unit_id']=OWNERS.unit_id.astype(str)
UNITS=UNITS.merge(OWNERS[['unit_type','unit_id','mosaic','roi_mosaic_coverage_fraction']],
 on=['unit_type','unit_id'],validate='one_to_one')
UNITS=UNITS[UNITS.roi_mosaic_coverage_fraction>=.95].copy()
assert len(UNITS)==106
display(UNITS[UNITS.mosaic.isin(SELECTED)].groupby(['site','unit_type'])
        .size().rename('footprints').reset_index())


## 03 — Deterministic inference readiness (no approval flag, no overview scan)

Every output pixel is checked against the actual ROI100, RGB, DSM and all four MS bands when its tile is processed.

In [ ]:
assert ROI_SETBACK_M==100.0
assert len(QA)==len(SELECTED)
assert all(QA.n_four_band_MS_flights>0)
print('Ready: strict 100 m interior ROI + RGB + DSM + four-band same-flight MS.')
print('Initial test:',SITE,'MAX_TILES:',MAX_TILES)


## 04 — Seam-suppressed RGB+MS inference (GPU)

For each 256px **output core**, request 64px additional sensor context in each direction, predict using that contextual patch, and write only the core. The model receives **real imagery outside the ROI interior as context**, but writes predictions *only* for core pixels passing the 100m inset, RGB, DSM and full same-flight MS gates. The patch is padded with zeros only where the RGB mosaic actually ends and is padded to a multiple of 32 for ResNet18 compatibility. A new filename signature prevents reuse of the old 256px edge-affected predictions.

No per-tile prediction averaging: cores are disjoint, and each pixel is predicted with substantial neighbourhood context. Source-flight selection is per tile and thus flight boundaries may retain spectral seams; if necessary, evaluate those separately.

In [ ]:
import torch
import segmentation_models_pytorch as smp
from rasterio.windows import bounds as window_bounds
assert torch.cuda.is_available(),'Select a GPU Colab runtime and rerun 00–03'
DEVICE='cuda'
def rgb_model_A(fold):
 nb03=V4/'03_paired_rgb_unet_v1'
 ck=torch.load(nb03/f'fold{fold}_A_{TRAIN["training_key"][:18]}.pt',
               map_location='cpu',weights_only=False)
 assert ck['training_key']==TRAIN['training_key']
 net=smp.Unet(encoder_name='resnet18',encoder_weights=None,in_channels=3,classes=1)
 net.load_state_dict({k.removeprefix('net.'):v for k,v in ck['model'].items()})
 return net
def ms_net(fold):
 base=rgb_model_A(fold).state_dict()
 net=smp.Unet(encoder_name='resnet18',encoder_weights=None,in_channels=7,classes=1)
 state=net.state_dict()
 for k,v in state.items():
  if k=='encoder.conv1.weight':
   v[:,:3]=base[k];v[:,3:]=0
  else:v.copy_(base[k])
 net.load_state_dict(state)
 ck=torch.load(MODEL_DIR/f'fold{fold}_RGB_MS_v1.pt',
               map_location='cpu',weights_only=False)
 assert ck['complete'] and ck['baseline_training_key']==TRAIN['training_key']
 net.load_state_dict(ck['state'],strict=True)
 return net.to(DEVICE).eval(),ck['fingerprint']
MODELS=[];MODEL_HASH=[]
for fold in range(3):
 model,h=ms_net(fold)
 MODELS.append(model);MODEL_HASH.append(str(h))
MODEL_SIGNATURE=hashlib.sha256(json.dumps(MODEL_HASH).encode()).hexdigest()[:12]
QA_SIGNATURE=hashlib.sha256(json.dumps(dict(
 schema='objective_ROI100_RGB_DSM_MS_v6_halo',setback=ROI_SETBACK_M,
 edge_halo_px=EDGE_HALO_PX,
 required=('RGB','DSM','red','green','red_edge','nir'),
 model=MODEL_SIGNATURE,registry=[
   (str(z.path),int(z.data_band),Path(z.path).stat().st_size,Path(z.path).stat().st_mtime_ns)
   for z in REG.itertuples()]),
 sort_keys=True).encode()).hexdigest()[:12]
print('NB05 RGB+MS model ensemble:',MODEL_SIGNATURE,'| coverage policy:',QA_SIGNATURE)
def make_vrts(ds,group,stack):
 result={}
 for b,(p,index) in bands_for(group).items():
  src=stack.enter_context(rasterio.open(p))
  vrt=stack.enter_context(WarpedVRT(src,crs=ds.crs,transform=ds.transform,
       width=ds.width,height=ds.height,resampling=Resampling.bilinear))
  result[b]=(vrt,index)
 return result
def read_ms_window(vrts,win):
 layers=[];valid=[]
 for b in ('red','green','red_edge','nir'):
  vrt,index=vrts[b]
  a=vrt.read(index,window=win,masked=True,out_dtype='float32')
  v=np.asarray(a.filled(0),np.float32)
  ok=(~np.ma.getmaskarray(a))&np.isfinite(v)&(v>0)
  layers.append(np.where(ok,v,0.));valid.append(ok)
 raw=np.stack(layers)
 valid=np.logical_and.reduce(valid)
 relative=np.where(valid[None],
        raw/np.maximum(raw.sum(axis=0,keepdims=True),1e-8),0.)
 return relative.astype('float32'),valid
def rgb_window(ds,win):
 a=ds.read([1,2,3],window=win,masked=True)
 assert all(np.dtype(t)==np.dtype('uint8') for t in ds.dtypes[:3])
 valid=(~np.ma.getmaskarray(a).any(axis=0))
 if ds.count>=4 and ds.colorinterp[3].name=='alpha':
  alpha=ds.read(4,window=win,masked=False)
  valid&=np.asarray(alpha)>0
 rgb=np.asarray(a.filled(0),np.float32)/255.
 return rgb,valid
def dsm_window(vrt,win):
 a=vrt.read(1,window=win,masked=True,out_dtype='float32')
 z=np.asarray(a.filled(np.nan),np.float32)
 return (~np.ma.getmaskarray(a))&np.isfinite(z)
def roi_window(ds,win,interior_geom):
 clip=interior_geom.intersection(box(*window_bounds(win,ds.transform)))
 if clip.is_empty:return np.zeros((int(win.height),int(win.width)),bool)
 return rasterize([(clip,1)],out_shape=(int(win.height),int(win.width)),
                  transform=ds.window_transform(win),fill=0,
                  dtype='uint8').astype(bool)
def tiles_for_roi(ds,interior):
 # Geometric tile discovery: no giant arrays, no missed narrow strips due
 # to a coarse sensor overview. Iterate only the inward ROI bounding range.
 if interior.is_empty:return []
 bb=rasterio.windows.from_bounds(*interior.bounds,transform=ds.transform)
 row0=max(0,int(np.floor(bb.row_off))//TILE)
 col0=max(0,int(np.floor(bb.col_off))//TILE)
 row1=min((ds.height-1)//TILE,int(np.ceil(bb.row_off+bb.height))//TILE)
 col1=min((ds.width-1)//TILE,int(np.ceil(bb.col_off+bb.width))//TILE)
 # Intersect ROI per tile, then check actual RGB/DSM/MS at full resolution
 # in run_mosaic. QA overview is not used for tile eligibility.
 return ((row,col) for row in range(row0,row1+1)
         for col in range(col0,col1+1)
         if interior.intersects(box(*rasterio.windows.bounds(
            Window(col*TILE,row*TILE,
              min(TILE,ds.width-col*TILE),min(TILE,ds.height-row*TILE)),
            ds.transform))))
def reference_centered_tiles(ds,interior,mosaic):
 # Choose complete test-footprint patches before exploring raster row order.
 # Reference selection does NOT use model output or measured Musanga cover.
 if not PRIORITIZE_REFERENCE_TILES or 'UNITS' not in globals():return []
 eligible=UNITS[UNITS.mosaic.eq(mosaic)].to_crs(ds.crs).copy()
 # favour circles and plots strictly inside eligible ROI and entirely on RGB
 eligible['interior_coverage']=[
   (g.intersection(interior).area/max(g.area,1e-9)) if g is not None and not g.is_empty else 0
   for g in eligible.geometry]
 eligible=eligible[eligible.interior_coverage>=.80].copy()
 eligible=eligible.sort_values(['unit_type','unit_id'],kind='stable')
 # Interleave plots and circles so each may appear in the atlas.
 groups=[eligible[eligible.unit_type.eq(k)] for k in ('circle','plot')]
 units=[]
 for records in zip(*[a.itertuples(index=False) for a in groups]):
  units.extend(records)
 # zip() may drop extra footprint types; append remaining sorted units.
 for row in eligible.itertuples(index=False):
  key=(row.unit_type,str(row.unit_id))
  if key not in {(q.unit_type,str(q.unit_id)) for q in units}:units.append(row)
 prioritized=[]
 included=0
 for z in units:
  if included>=MAX_PRIORITY_FOOTPRINTS_PER_MOSAIC:break
  # Slight halo: include full footprint bounding box and 1 tile margin
  bb=rasterio.windows.from_bounds(*z.geometry.bounds,transform=ds.transform)
  row0=max(0,int(np.floor(bb.row_off))//TILE-1)
  col0=max(0,int(np.floor(bb.col_off))//TILE-1)
  row1=min((ds.height-1)//TILE,int(np.ceil(bb.row_off+bb.height))//TILE+1)
  col1=min((ds.width-1)//TILE,int(np.ceil(bb.col_off+bb.width))//TILE+1)
  candidate=[(row,col) for row in range(row0,row1+1)
              for col in range(col0,col1+1)
              if interior.intersects(box(*rasterio.windows.bounds(
                Window(col*TILE,row*TILE,
                       min(TILE,ds.width-col*TILE),min(TILE,ds.height-row*TILE)),
                ds.transform)))]
  if not candidate:continue
  prioritized.extend(candidate)
  included+=1
  print('  Prioritizing',z.unit_type,z.unit_id,
        'ROI coverage',round(float(z.interior_coverage),2),
        '| tiles',len(candidate),flush=True)
 return list(dict.fromkeys(prioritized))
def core_and_context_window(ds,row,col):
 # Read only actual mosaic pixels; at edges zero-pad to a /32 input size.
 # Core georeferencing is ALWAYS determined from the original unpadded tile.
 r0=row*TILE;c0=col*TILE
 h=min(TILE,ds.height-r0);w=min(TILE,ds.width-c0)
 core=Window(c0,r0,w,h)
 context=Window(max(0,c0-EDGE_HALO_PX),max(0,r0-EDGE_HALO_PX),
    min(ds.width,c0+w+EDGE_HALO_PX)-max(0,c0-EDGE_HALO_PX),
    min(ds.height,r0+h+EDGE_HALO_PX)-max(0,r0-EDGE_HALO_PX))
 inner_r=int(r0-context.row_off);inner_c=int(c0-context.col_off)
 return core,context,(inner_r,inner_c,h,w)
def padded_context_features(rgb,ms):
 # Keep source alignment unchanged, pad only bottom/right to /32.
 h,w=rgb.shape[-2:]
 hh=int(np.ceil(h/32)*32);ww=int(np.ceil(w/32)*32)
 x=np.zeros((7,hh,ww),np.float32)
 x[:3,:h,:w]=rgb
 x[3:,:h,:w]=ms
 return x
@torch.inference_mode()
def run_mosaic(mosaic):
 manifest=[]
 mdir=TILE_DIR/mosaic.replace('.tif','')
 mdir.mkdir(parents=True,exist_ok=True)
 saved=0
 with ExitStack() as stack:
  rgb=stack.enter_context(rasterio.open(MOSAICS[mosaic]))
  dsm=stack.enter_context(rasterio.open(DSM_BY_MOSAIC[mosaic]))
  dsm_vrt=stack.enter_context(WarpedVRT(
       dsm,crs=rgb.crs,transform=rgb.transform,
       width=rgb.width,height=rgb.height,resampling=Resampling.bilinear))
  interior=gpd.GeoSeries([ROI_INTERIOR],crs='EPSG:32635').to_crs(rgb.crs).iloc[0]
  flights={g:make_vrts(rgb,g,stack) for g,owner in GROUP_MOSAIC.items()
           if owner==mosaic}
  priority=reference_centered_tiles(rgb,interior,mosaic)
  from itertools import chain
  priority_set=set(priority)
  def remaining():
   for tile in tiles_for_roi(rgb,interior):
    if tile not in priority_set:yield tile
  locations=chain(priority,remaining())
  print(mosaic,'| halo:',EDGE_HALO_PX,'px | prioritized tiles:',len(priority),flush=True)
  for row,col in locations:
   core,context,(ir,ic,h,w)=core_and_context_window(rgb,row,col)
   # Evaluate acceptance only on the exact 256px output core.
   roi_core=roi_window(rgb,core,interior)
   if not roi_core.any():continue
   dest=mdir/f'qa{QA_SIGNATURE}_m{MODEL_SIGNATURE}_halo{EDGE_HALO_PX:03d}_r{row:05d}_c{col:05d}.tif'
   if dest.is_file():
    # Never reuse a corrupt or incomplete GeoTIFF.
    try:
     with rasterio.open(dest) as chk:
      if chk.count!=2 or chk.width!=w or chk.height!=h or chk.nodata!=-9999.:
       raise ValueError('cached output geometry invalid')
      if str(chk.tags().get('edge_halo_px'))!=str(EDGE_HALO_PX):
       raise ValueError('cached output halo mismatch')
    except Exception:
     print('Invalid cached tile, regenerating:',dest.name,flush=True)
     dest.unlink(missing_ok=True)
    else:
     manifest.append(dict(mosaic=mosaic,row=row,col=col,path=str(dest),
                    status='reused',sensor_flight='cached'))
     saved+=1
     if MAX_TILES is not None and saved>=MAX_TILES:break
     continue
   rgb_context,rgb_ok=rgb_window(rgb,context)
   dsm_ok=dsm_window(dsm_vrt,context)
   core_basic=roi_core & rgb_ok[ir:ir+h,ic:ic+w] & dsm_ok[ir:ir+h,ic:ic+w]
   if not core_basic.any():continue
   candidates=[]
   for flight,vrts in flights.items():
    ms_context,ms_ok=read_ms_window(vrts,context)
    n=int(np.count_nonzero(core_basic&ms_ok[ir:ir+h,ic:ic+w]))
    candidates.append((n,flight,ms_context,ms_ok))
   if not candidates:continue
   nvalid,flight,ms_context,ms_ok=max(candidates,key=lambda z:z[0])
   valid=core_basic & ms_ok[ir:ir+h,ic:ic+w]
   if nvalid==0 or valid.mean()<MIN_TILE_QA_FRACTION:continue
   features=padded_context_features(rgb_context,ms_context)
   tensor=torch.from_numpy(features[None]).to(DEVICE)
   core_predictions=[]
   for net in MODELS:
    with torch.autocast('cuda'):
     full=torch.sigmoid(net(tensor)[:,0]).float()[0].cpu().numpy()
    core_predictions.append(full[ir:ir+h,ic:ic+w])
   prob=np.mean(core_predictions,axis=0).astype(np.float32)
   prob[~valid]=-9999.
   binary=np.where(valid,(prob>=PROB_THRESHOLD).astype('float32'),-9999.)
   # Clean profile: do not inherit raster block size from source imagery.
   profile=dict(driver='GTiff',count=2,height=h,width=w,crs=rgb.crs,
     transform=rgb.window_transform(core),dtype='float32',
     nodata=-9999.,compress='deflate',tiled=False,interleave='band')
   tmp=dest.with_name(dest.stem+'.temp.tif')
   with rasterio.open(tmp,'w',**profile) as out:
    out.write(prob,1);out.write(binary,2)
    out.set_band_description(1,'Musanga probability, 64px contextual inference')
    out.set_band_description(2,'Musanga binary at probability threshold')
    out.update_tags(ms_flight=flight,roi_setback_m=ROI_SETBACK_M,
       edge_halo_px=EDGE_HALO_PX,model_folds='0,1,2',qa_signature=QA_SIGNATURE)
   os.replace(tmp,dest)
   manifest.append(dict(mosaic=mosaic,row=row,col=col,path=str(dest),
              status='new',sensor_flight=flight,valid_fraction=float(valid.mean())))
   saved+=1
   if saved%10==0:print('Completed',mosaic,saved,'tiles',flush=True)
   if MAX_TILES is not None and saved>=MAX_TILES:break
 return manifest

TILE_MANIFEST=[]
for mosaic in SELECTED:TILE_MANIFEST.extend(run_mosaic(mosaic))
TILE_MANIFEST=pd.DataFrame(TILE_MANIFEST)
TILE_MANIFEST.to_csv(OUT/'04_prediction_tile_manifest.csv',index=False)
print('Inference tiles saved/reused:',len(TILE_MANIFEST))
if len(TILE_MANIFEST):
 display(TILE_MANIFEST.groupby(['mosaic','status']).size().rename('tiles').reset_index())
else:print('No eligible tiles. Check ROI interior and source coverage tables.')


## 05 — Six-panel atlas with high-visibility reference annotation outlines

White, black-cased **individual crown annotation boundaries** are overlaid on panels 3–6, including the probability, binary and red-on-RGB predictions. The yellow line is **the reference footprint boundary**, not an individual crown. Predictions and masks are unchanged by plotting. The atlas only uses the current halo prediction tile manifest (not old uncropped files).

In [ ]:
import matplotlib.colors as mcolors
import matplotlib.patheffects as pe
from rasterio.merge import merge
# To keep figures honest, use only units actually intersecting saved prediction
# tiles and require >=50% quality-approved support within the shown unit.
assert 'TILE_MANIFEST' in globals() and len(TILE_MANIFEST)>0, 'Run Section 04 first. No accepted inference tiles exist.'
PLOT_SOURCE=gpd.read_file(F78['exhaustive_plot_crowns']).to_crs('EPSG:32635')
PLOT_SOURCE['positive']=pd.to_numeric(PLOT_SOURCE.Class_ID,errors='coerce').eq(1)
CIRCLE_SOURCE=gpd.read_file(F78['circle_positive_crowns']).to_crs('EPSG:32635')
CIRCLE_SOURCE['positive']=True
def plot_a_few():
 used=[]
 for mosaic,files in TILE_MANIFEST.groupby('mosaic'):
  units=UNITS[UNITS.mosaic.eq(mosaic)].copy()
  if not len(units):continue
  tilefiles=list(files.path)
  # Reference footprints can straddle tile boundaries. Select one footprint
  # whose whole buffered bounding box intersects saved tiles; mosaic merge
  # below reconstructs an excerpt from the georeferenced probability tiles.
  with ExitStack() as stack:
   src=[stack.enter_context(rasterio.open(p)) for p in tilefiles]
   union_bounds=unary_union([box(*p.bounds) for p in src])
   with rasterio.open(MOSAICS[mosaic]) as rgb:
    with ExitStack() as vst:
     group=next((g for g,m in GROUP_MOSAIC.items() if m==mosaic),None)
     vrts=make_vrts(rgb,group,vst)
     for z in units.to_crs(rgb.crs).itertuples():
      if not union_bounds.intersects(z.geometry):continue
      win=from_bounds(*z.geometry.bounds,transform=rgb.transform)
      c0=max(0,int(np.floor(win.col_off)));r0=max(0,int(np.floor(win.row_off)))
      c1=min(rgb.width,int(np.ceil(win.col_off+win.width)))
      r1=min(rgb.height,int(np.ceil(win.row_off+win.height)))
      ww=c1-c0;hh=r1-r0
      if ww<15 or hh<15 or ww*hh>8_000_000:continue
      window=Window(c0,r0,ww,hh)
      ptiles=[q for q in src if box(*q.bounds).intersects(z.geometry)]
      if not ptiles:continue
      pred,tr=merge(ptiles,bounds=rasterio.windows.bounds(window,rgb.transform),
                    nodata=-9999.,indexes=[1],res=(abs(rgb.transform.a),abs(rgb.transform.e)))
      p=np.asarray(pred[0],float)
      if p.shape!=(hh,ww):continue
      refmask=rasterize([(z.geometry,1)],out_shape=(hh,ww),
               transform=rgb.window_transform(window),fill=0).astype(bool)
      good=(p>=0)&(p<=1)
      if not refmask.any() or good[refmask].mean()<.20:continue
      rgb3,rv=rgb_window(rgb,window)
      ms,msv=read_ms_window(vrts,window)
      false=np.stack([ms[3],ms[2],ms[0]],axis=-1)
      validMS=msv&(ms.sum(axis=0)>0)
      if validMS.any():
       for band in range(3):
        q=false[:,:,band]
        lo,hi=np.percentile(q[validMS],[2,98])
        false[:,:,band]=np.clip((q-lo)/max(hi-lo,1e-8),0,1)
      false[~validMS]=1
      extent=rasterio.windows.bounds(window,rgb.transform)
      extent=[extent[0],extent[2],extent[1],extent[3]]
      fig,axes=plt.subplots(1,6,figsize=(24,4.2),layout='constrained')
      axes[0].imshow(np.moveaxis(rgb3,0,-1),extent=extent,origin='upper')
      axes[1].imshow(false,extent=extent,origin='upper')
      axes[2].imshow(np.moveaxis(rgb3,0,-1),extent=extent,origin='upper')
      src_gdf=CIRCLE_SOURCE if z.unit_type=='circle' else PLOT_SOURCE
      geom32635=gpd.GeoSeries([z.geometry],crs=rgb.crs).to_crs('EPSG:32635').iloc[0]
      selected=src_gdf.iloc[src_gdf.sindex.query(geom32635,predicate='intersects')].copy()
      if len(selected):
       selected=selected.to_crs(rgb.crs)
      probability=np.ma.masked_where(~good,p)
      axes[3].imshow(probability,extent=extent,origin='upper',
                     cmap='magma',vmin=0,vmax=1,interpolation='nearest')
      binary=np.ma.masked_where(~good,(p>=PROB_THRESHOLD).astype(float))
      axes[4].imshow(binary,extent=extent,origin='upper',
                     cmap='Greens',vmin=0,vmax=1,interpolation='nearest')
      axes[5].imshow(np.moveaxis(rgb3,0,-1),extent=extent,origin='upper')
      # Saturated red prediction is intentionally stronger than old spring overlay.
      predmask=np.ma.masked_where(~good|(p<PROB_THRESHOLD),np.ones_like(p))
      redmap=mcolors.ListedColormap(['#ff2735'])
      axes[5].imshow(predmask,extent=extent,origin='upper',
                     cmap=redmap,vmin=0,vmax=1,alpha=.60,interpolation='nearest')
      # Actual reference crowns, NOT the footprint bounding mask:
      # thick white lines with thin dark casing for contrast on all backgrounds.
      for ax in (axes[2],axes[3],axes[4],axes[5]):
       if len(selected):
        for polygon in selected.geometry:
         if polygon is None or polygon.is_empty:continue
         # GeoPandas border artists apply to Polygon and MultiPolygon.
         group=gpd.GeoSeries([polygon],crs=rgb.crs)
         before=len(ax.collections)
         group.boundary.plot(ax=ax,color='white',linewidth=1.65,zorder=15)
         for artist in ax.collections[before:]:
          artist.set_path_effects([pe.Stroke(linewidth=2.8,foreground='#101010'),
                                   pe.Normal()])
      for ax,title in zip(axes,['RGB','MS NIR/RE/R','Annotations','Probability',
                                 'Binary Musanga','Prediction overlay']):
       gpd.GeoSeries([z.geometry],crs=rgb.crs).boundary.plot(
          ax=ax,color='#ffec4d',linewidth=1.4)
       ax.set(xlim=(extent[0],extent[1]),ylim=(extent[2],extent[3]),title=title)
       ax.set_xticks([]);ax.set_yticks([])
      fig.suptitle(f'{z.site} {z.unit_type} {z.unit_id} | ROI100 + all sensors; valid={good[refmask].mean():.0%} | halo={EDGE_HALO_PX}px')
      out=FIG/f'example_halo{EDGE_HALO_PX:03d}_{mosaic.replace(".tif","")}_{z.unit_type}_{z.unit_id}.png'
      fig.savefig(out,dpi=175,bbox_inches='tight');plt.show();plt.close(fig)
      used.append(dict(site=z.site,unit_type=z.unit_type,unit_id=z.unit_id,
                       path=str(out),approved_fraction=float(good[refmask].mean())))
      if len(used)>=6:return used
 return used
EXAMPLES=pd.DataFrame(plot_a_few())
EXAMPLES.to_csv(OUT/'05_example_atlas_manifest.csv',index=False)
print('Completed QA example panels:',len(EXAMPLES))
display(EXAMPLES)
if not len(EXAMPLES):
 print('No reference footprint passed the strict overlap/ROI/sensor tests. '
       'Displaying genuine saved inference tiles instead (not validation footprints).')
 fig,axs=plt.subplots(1,3,figsize=(12,4),layout='constrained')
 shown=0
 for q in TILE_MANIFEST.itertuples(index=False):
  with rasterio.open(q.path) as src:
   p=src.read(1,masked=True)
   valid=~np.ma.getmaskarray(p)
   if not np.any(valid):continue
   axs[shown].imshow(p,cmap='magma',vmin=0,vmax=1)
   axs[shown].set_title(f'{q.mosaic} tile r{q.row} c{q.col}\nvalid={valid.mean():.0%}')
   axs[shown].axis('off')
   shown+=1
   if shown==3:break
 for ax in axs[shown:]:ax.axis('off')
 if shown:
  fig.savefig(FIG/f'05_fallback_prediction_tile_preview_halo{EDGE_HALO_PX:03d}.png',
              dpi=160,bbox_inches='tight')
  plt.show()
 plt.close(fig)
